In [8]:
from scipy.sparse import load_npz
import os
import pickle
import implicit
import pandas as pd
import kagglehub
import numpy as np


In [9]:
DATA_DIR = os.path.join("..", "data")  

train = pd.read_parquet(os.path.join(DATA_DIR, "train_ratings.parquet"))
test = pd.read_parquet(os.path.join(DATA_DIR, "test_ratings.parquet"))

anime_ids = train['anime_id'].astype('category')
anime_id_map = dict(enumerate(anime_ids.cat.categories)) 
anime_id_map_reverse = {v: k for k, v in anime_id_map.items()} 

user_ids = train['user_id'].astype('category')
user_id_map = dict(enumerate(user_ids.cat.categories))
user_id_map_reverse = {v: k for k, v in user_id_map.items()}

train['anime_idx'] = anime_ids.cat.codes
train['user_idx'] = user_ids.cat.codes

test['anime_idx'] = test['anime_id'].map(anime_id_map_reverse)
test['user_idx'] = test['user_id'].map(user_id_map_reverse)

path = kagglehub.dataset_download("ramazanturann/user-animelist-dataset")
animes = pd.read_csv(os.path.join(path, "animes.csv"))

In [2]:
item_user_matrix = load_npz(os.path.join(DATA_DIR, "item_user_matrix.npz"))

user_item_matrix = item_user_matrix.T.tocsr()

model = implicit.als.AlternatingLeastSquares(factors=64, regularization=0.01, iterations=20)
model.fit(user_item_matrix)

with open(os.path.join(DATA_DIR, "als_model.pkl"), "wb") as f:
    pickle.dump(model, f)

  0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
def als_recommend_for_user(user_idx, k=10):
    recommended = model.recommend(
        user_idx,
        user_item_matrix[user_idx],  # this user's row — used to exclude already-seen items
        N=k
    )
    item_indices, scores = recommended
    return [anime_id_map[i] for i in item_indices]

sample_uid = 5
recs = als_recommend_for_user(sample_uid, k=10)
print(animes[animes['animeID'].isin(recs)][['animeID', 'title']])



      animeID                                    title
11         12                             Cowboy Bebop
12         13                      Fullmetal Alchemist
37         38                         Samurai Champloo
47         48                            Dragon Ball Z
100       101                             Vinland Saga
102       103                                  Gintama
147       148  Code Geass: Lelouch of the Rebellion R2
282       283                    Great Teacher Onizuka
408       409                                  Clannad
1186     1187                            Steins;Gate 0


In [ ]:
def precision_recall_als(test_df, k=10, sample_users=None):
    total_relevant = 0
    total_recommended_relevant = 0
    
    test_positive = test_df[test_df['is_positive'] == 1]
    grouped = test_positive.groupby('user_idx')
    
    users_to_eval = list(grouped.groups.keys())
    if sample_users:
        users_to_eval = np.random.choice(users_to_eval, size=sample_users, replace=False)
    
    for user_idx in users_to_eval:
        group = grouped.get_group(user_idx)
        actual_positive = set(group['anime_id'])
        
        recommended = set(als_recommend_for_user(user_idx, k=k))
        
        hits_this_user = len(actual_positive & recommended)
        total_recommended_relevant += hits_this_user
        total_relevant += len(actual_positive)
    
    recall = total_recommended_relevant / total_relevant if total_relevant > 0 else 0
    precision = total_recommended_relevant / (len(users_to_eval) * k)
    
    return precision, recall

test = test[test['is_positive'] == 1].dropna(subset=['anime_idx', 'user_idx'])

precision, recall = precision_recall_als(test, k=10, sample_users=2000)
print(f"ALS — Precision@10: {precision:.4f}")
print(f"ALS — Recall@10: {recall:.4f}")

KeyError: ['anime_idx', 'user_idx']